In [ ]:
import pandas as pd 
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt 
from PIL import Image
from pathlib import Path

In [ ]:
# filter seams for global flat weren't interpolated but flat application happens after filter seam interpolation of the obs data
# and the filter seams for the target flat were interpolated. so we are going to linearly interpolate the global flat seams in the spectral 
# direction here. 

In [ ]:
import numpy as np

def interpolate_rows(arr, row_starts, row_counts=None):
    """
    Linearly interpolates over specified rows using the rows above and below.

    Parameters
    ----------
    arr        : 2D numpy array
    row_starts : int or list of ints - first row (or only row) of each bad region
    row_counts : int or list of ints - number of consecutive rows to interpolate
                 per region. Defaults to 1 for each region if not provided.

    Returns
    -------
    Interpolated array (float copy of input)
    """
    
    if np.isscalar(row_starts):
        row_starts = [row_starts]
    if row_counts is None:
        row_counts = [1] * len(row_starts)
    elif np.isscalar(row_counts):
        row_counts = [row_counts] * len(row_starts)

    for start, count in zip(row_starts, row_counts):
        top_row    = start - 1          # row above
        bottom_row = start + count      # row below
        n_steps    = count + 1          # how many rows to do 

        if top_row < 0 or bottom_row >= arr.shape[0]:
            raise ValueError(
                f"Region starting at row {start} with count {count} "
                f"has no valid boundary rows (array has {arr.shape[0]} rows)."
            )

        top    = arr[top_row, :]
        bottom = arr[bottom_row, :]

        for i, row in enumerate(range(start, start + count)):
            t = (i + 1) / n_steps     
            arr[row, :] = top + t * (bottom - top)

    return arr


In [ ]:
path = "/home/bekah/m3-pipeline-dev/cal_files/lab_flat_field_global.fits"

with fits.open(path, memmap=True) as hdul:
    data = hdul[0].data

# 0 indexed so 13 and 50 are 12 and 49 
result = interpolate_rows(data, row_starts=[12, 49], row_counts=[1, 1])


In [ ]:
fits.writeto("/home/bekah/m3-pipeline-dev/cal_files/lab_flat_field_global_interpolated.fits", result, overwrite=True)

In [ ]:
# diff them 

path = "/home/bekah/m3-pipeline-dev/cal_files/lab_flat_field_global.fits"

with fits.open(path, memmap=True) as hdul:
    old = hdul[0].data

path = "/home/bekah/m3-pipeline-dev/cal_files/lab_flat_field_global_interpolated.fits"

with fits.open(path, memmap=True) as hdul:
    new = hdul[0].data

diff = old - new 

In [ ]:
fits.writeto("/home/bekah/m3-pipeline-dev/cal_files/diff_new_old_global_flat.fits", diff, overwrite=True)